# Research Log: Framing the Game

Running notebook for brainstorms, hypotheses, experimental results, and reviewer-driven insights.

**Paper**: *Framing the Game: How Context Shapes LLM Decision-Making*  
**Authors**: Isaac Robinson, John Burden  
**Target**: ICLR 2026 resubmission  

---

## Key Reviewer Feedback (ICLR 2026 Round 1)

**Result**: Reject (scores: 4, 2, 6, 4)

### Must-address concerns
1. **Generalizability beyond one-shot PD** — all four reviewers flagged this. Need to either extend to other games or better justify the PD-only scope.
2. **LLM-as-judge QC without human validation** — meta-reviewer listed this as a primary rejection reason. Need human eval or at minimum judge-swap analysis.
3. **Descriptive, not mechanistic** — reviewers wanted to know *why* certain contexts drive cooperation, not just *that* they do (mVh3, aAtL).
4. **Limited reasoning model coverage** — mQG5 specifically flagged this; only 2 small R1 distillations tested.

### Addressable in resubmission
- Better topic selection to tell a clearer story
- 108-model registry now covers full reasoning model spectrum (o1, o3, R1, QwQ)
- Need behavioral/mechanistic analysis connecting topics to cooperation rates
- Need human validation study for vignette QC

---

## Brainstorms & Hypotheses

*(newest first)*

### 2026-02-20: Topic Redesign Brainstorm

**Problem**: Original 10 topics were redundant (5/10 politics, 2 generic) and produced results that were descriptive but not compelling. Reviewers asked "why" and we couldn't answer.

**Insight**: Topics should be chosen so the cooperation gradient is *predictable a priori* from domain norms, making the finding interpretable rather than just descriptive.

#### Promising axes of variation

1. **Moral valence of cooperation** — cooperation can be prosocial (sharing), neutral (trade), or antisocial (collusion, cover-up). If models cooperate less when cooperation is "wrong", that's evidence of learned moral reasoning overriding game theory.

2. **Cultural/geographic framing** — same scenario with different cultural context (Silicon Valley vs Tokyo vs Lagos). Would reveal training-data cultural stereotypes in strategic behavior. High bias/fairness relevance.

3. **Power asymmetry** — large corp vs startup, employer vs employee. Tests whose interests models default to.

4. **Observability** — private vs public negotiation. In one-shot PD, observability shouldn't matter. If it does, models are importing reputation-game logic.

5. **Stakes magnitude** — neighborhood dispute vs international crisis. Tests stake-sensitivity.

6. **Political dyads** — specific party matchups (R vs D, D vs Green, bipartisan committee). Tests whether models reflect real-world partisan polarization.

7. **Temporal/historical distance** — same dilemma in 1400 vs 1940 vs 2025. Models may grant "moral license" to defect in historical settings.

#### Open question
Which 2-3 axes to combine? The interaction effects (e.g., "cultural framing matters more in business than medicine") are where the paper gets strongest.

### 2026-02-20: Axes Selected [design]

**Selected 6 axes** (dropped stakes magnitude):
1. **Moral valence of cooperation** — prosocial vs neutral vs antisocial cooperation
2. **Cultural/geographic framing** — same scenario, different cultural context
3. **Power asymmetry** — symmetric vs asymmetric actor power
4. **Observability** — private vs public (shouldn't matter in one-shot PD but probably does)
5. **Political dyads** — specific party matchups with known real-world antagonism levels
6. **Temporal/historical distance** — same dilemma across time periods

**Next step**: Combine into a concrete topic set (10-13 topics) that covers multiple axes without being unwieldy. Key challenge is that 6 axes fully crossed would be enormous — need to pick topics that naturally embed 1-2 axes each.

### 2026-02-20: Power Analysis Results [result]

**Setup**: Two-proportion z-test, α=0.05, power=0.80, baseline coop ~55%

**Key takeaways**:
- To detect a **30pp difference** (e.g. moral valence extremes): only **~42 stories/group** needed — very cheap
- To detect a **15pp difference** (e.g. political dyads, temporal): **~170 stories/group** — moderate
- To detect a **10pp difference** (e.g. cultural framing): **~390 stories/group** — expensive
- To detect a **5pp difference**: ~1,550/group — probably not worth it

**Implications for axis selection**:
- **Moral valence** (expected 30pp effect): easiest to power, ~126 stories total. Do this.
- **Political dyads** (expected 15-25pp): ~250-700 total across 4 groups. Feasible.
- **Temporal distance** (expected 10-20pp): ~290-1,164 total. Feasible if effect is large enough.
- **Observability** (binary, expected 10-20pp): ~192-540 total. Cheap because only 2 groups.
- **Power asymmetry** (binary, expected 10-20pp): ~194-540 total. Same.
- **Cultural/geographic** (expected 5-15pp): **riskiest** — if effect is only 5-10pp, need 1,500+ stories/group. Could be underpowered unless effect is surprisingly large.

**Budget**: At $0.62/sweep for all 108 models, 200 stories/group × 10 groups = $1,240 total. Manageable.

### 2026-02-20: Budget approved [design]

**$1,200 budget approved** for full 108-model sweep at ~200 stories/group × ~10 groups. All 6 axes are a go including cultural/geographic (will accept risk of underpowering if effect is small).

**Next step**: Design the concrete topic set and config structure — which axes become topics vs config dimensions.

### 2026-02-20: Experimental Design Decisions [design]

**Structure**: topics × actor_types × observability × power_dynamic

**Decisions made**:
- **Drop `neutral` actor type** — keep only allies vs enemies for cleaner contrast
- **Drop `world_type`** (real vs imaginary) — wasn't a strong finding in v1, doubles cell count for little value
- **Add `observability`**: private vs public (binary)
- **Add `power_dynamic`**: symmetric vs asymmetric (binary)
- **Budget**: uncapped for now, prioritize at least 6 topics per axis category
- **Target**: ≥6 topics per axis (moral valence, political dyads, temporal, cultural/geographic)

**Cross product**: N topics × 2 actors × 2 observability × 2 power = N × 8 cells per topic

### 2026-02-20: External Brainstorm Synthesis [brainstorm]

**Key refinements from external model feedback:**

1. **Temporal axis**: Hold the *type* of interaction constant (e.g., "two powers negotiating a trade pact"), vary ONLY time period. Avoids confounding substance with era.
2. **Moral valence**: Ensure antisocial cooperation is explicitly illegal/exploitative, not just morally gray. Clean 3-bin structure (prosocial / neutral / antisocial).
3. **Political dyads**: Expand to include international adversaries (US vs China, US vs EU) for a fuller ideological-distance gradient. Monotonic prediction: cooperation ∝ perceived affinity.
4. **Cultural**: Structure by Hofstede-style dimensions (collectivist vs individualist, high-trust vs low-trust). Add Nordic (Stockholm) for high-trust baseline.
5. **Orthogonality**: Don't let moral valence confound actor_type axis. "Rival gangs" already implies enemies + antisocial — avoid this.
6. **Track refusals**: Model refusing to engage with antisocial cooperation is itself a finding.
7. **Explicit hypotheses**: H1 (moral valence), H2 (ideological distance), H3 (temporal modernity), H4 (cultural trust proxies).
8. **Optional 5th axis**: Anthropomorphism (individuals vs corporations vs AI systems vs governments) — interesting but parking for now.

### 2026-02-20: Temporal axis redesign — zoom into modernity [design]

**Insight**: The interesting question isn't "ancient vs modern" — it's whether models reflect shifting norms *within* recent history. Training data is 1000x denser for 2000s-2020s than for 1200 BCE. Cultural inflection points (social media, 2016 polarization, COVID) may produce measurable cooperation shifts even across single decades.

**New design**: 1 ancient anchor + dense modern granularity. Same underlying scenario: "two sovereign powers negotiating a trade agreement."

Proposed eras:
- ~1200 BCE (Bronze Age anchor — Hobbesian baseline)
- 1850s (Industrial, pre-world-wars, imperial competition)
- 1940s (WWII/postwar, institutional cooperation born out of catastrophe)
- 1970s (détente, Cold War thaw, pragmatic cooperation)
- early 2000s (post-9/11, "war on terror," multilateralism strained)
- 2010s (pre-Trump, peak globalization consensus)  
- 2020s (COVID, polarization peak, institutional distrust)
- near-future (speculative — tests whether models default to optimism or dystopia)

**Hypothesis refinement**: Cooperation may NOT be monotonically increasing with time. Possible non-linear pattern:
- 1940s bump (postwar institution-building)
- 2000s dip (post-9/11 unilateralism)
- 2020s dip (polarization, distrust)
- Near-future uncertain (utopian vs dystopian training data)

This non-linearity would be a much more interesting finding than a simple "modern = more cooperative" gradient.

### 2026-02-20: Temporal axis — tighter parallel structure [design]

**Problem**: Current temporal topics vary both the era AND the scenario type (grain trade, rail access, reconstruction aid, pandemic response). This confounds time with substance — if cooperation differs between "1940s reconstruction aid" and "2020s pandemic response," is that because of the era or because aid ≠ pandemic?

**Fix**: Use the exact same scenario template for all time periods. Only the date and period-appropriate nouns change. E.g., "two leaders negotiating a trade agreement in [era]" — same actors (political leaders), same action (trade negotiation), same stakes.

### 2026-02-21: Design rigor audit [design]

Reviewing the 30-topic design for confounds, missing controls, and methodological gaps before implementation.

#### Issues identified:

**1. Moral valence axis is confounded by domain**
"Hospitals sharing ventilators" vs "banks manipulating interest rates" differs in moral valence AND domain, actors, stakes, and familiarity. Any cooperation difference could be healthcare vs finance, not prosocial vs antisocial. 

**Fix**: Use matched pairs within the SAME domain where cooperation flips moral meaning:
- "Two pharma companies sharing drug trial data" (prosocial) vs "Two pharma companies coordinating drug pricing" (antisocial)
- Same actors, same industry — only the moral meaning of cooperation changes.

**2. No abstract baseline control**
Without a "naked PD" with no narrative framing, we can't measure how much ANY context shifts behavior. We can only say "X has more cooperation than Y," not "X increases cooperation by N pp over the rational baseline."

**3. Position counterbalancing**
If "cooperate" is always option A, models might show position bias. Need to randomize A/B assignment.

**4. Observability and power dimensions need concrete prompt specifications**
How exactly do these get injected into vignettes? Need standardized phrasing.

**5. No holdout set defined for the new design**

**6. Political dyads axis also varies scenario substance**
"Bipartisan Senate committee" vs "US vs China export controls" differs in both ideological distance AND institutional setting. Could hold setting constant: "negotiating a policy agreement" and vary only which parties/nations.

### 2026-02-21: Final topic set implemented [design]

All 6 methodological fixes from the rigor audit have been applied and implemented in `config.py`.

**Structure**: 43 topics (35 main + 8 holdout) across 5 axes, crossed with 3 binary dimensions.

#### Axes and topic counts

| Axis | Main | Holdout | Design principle |
|---|---|---|---|
| **Moral valence** | 12 (6 matched pairs) | 2 | Same domain, cooperation flips moral meaning |
| **Political dyads** | 8 | 2 | Same template ("negotiating a policy agreement"), vary only parties |
| **Temporal distance** | 8 | 2 | Same scenario ("two national leaders negotiating a trade agreement in [era]"), vary only era |
| **Cultural/geographic** | 6 | 2 | Same scenario ("two business executives negotiating a joint venture in [city]"), vary only location |
| **Baseline** | 1 | 0 | Abstract PD, no narrative framing |

#### Cross-cutting dimensions (280 total cells)
- **actor_type**: allies / enemies
- **observability**: private / public (with standardized prompt injection text)
- **power_dynamic**: symmetric / asymmetric (with standardized prompt injection text)

#### Methodological fixes applied
1. **Moral valence deconfounded** — matched pairs within same industry (pharma, tech, finance, agriculture, real estate, shipping)
2. **Abstract baseline added** — naked PD control for measuring effect of ANY narrative framing
3. **Political dyads parallel structure** — uniform template, only party names change
4. **Position counterbalancing** — noted for generator implementation (A/B swap already exists in analysis)
5. **Concrete dimension prompts** — `OBSERVABILITY` and `POWER_DYNAMIC` dicts have standardised phrasing
6. **Holdout set defined** — 2 topics per axis (8 total) for validation

#### Key hypotheses (testable)
- **H1**: Cooperation rate drops when cooperation = antisocial (moral valence)
- **H2**: Cooperation ∝ perceived ideological affinity (political dyads)
- **H3**: Non-linear temporal pattern — 1940s bump, 2000s/2020s dips (temporal)
- **H4**: Cultural trust proxies predict cooperation (Stockholm > Tokyo > Lagos) (cultural)